In [1]:
import pandas as pd

import os
import glob
for dirname, _, filenames in os.walk('/spam-detection'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
excel_files = glob.glob("*.xlsx")
excel_files

['2024- text comments.xlsx.xlsx',
 'Cycle 1- text comments.xlsx.xlsx',
 'JAN 2025- text comments.xlsx.xlsx',
 'MAR 2025 Reporting- text comments.xlsx.xlsx']

In [3]:
data = pd.concat([pd.read_excel(file) for file in excel_files]).reset_index()

In [ ]:
# Fill NaN values in 'comments' with empty string
data.fillna({"comments": ""}, inplace=True)

In [5]:
data.dtypes

index                               int64
respondent id                       int64
question code                      object
comments                           object
hide comment                         bool
language id                        object
edited text                       float64
english translation               float64
positive sentiment probability    float64
neutral sentiment probability     float64
negative sentiment probability    float64
sentiment category                 object
dtype: object

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score
import re

In [7]:
def clean_text(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = text.lower().strip()
    return text

In [8]:
data.columns

Index(['index', 'respondent id', 'question code', 'comments', 'hide comment',
       'language id', 'edited text', 'english translation',
       'positive sentiment probability', 'neutral sentiment probability',
       'negative sentiment probability', 'sentiment category'],
      dtype='object')

In [9]:
# data['comments'] = data['comments'].apply(clean_text)
X = data[['comments', 'sentiment category']]
y = data['hide comment']
X = pd.get_dummies(X, columns=['sentiment category'])

# Vectorize 'comments' column using TF-IDF
vectorizer = TfidfVectorizer()
comments_vectorized = vectorizer.fit_transform(X['comments']).toarray()

# Drop the original 'comments' column and add the TF-IDF features
X = X.drop(columns=['comments'])
X = pd.concat([X, pd.DataFrame(comments_vectorized)], axis=1)

In [10]:
# Fill any remaining NaN values in X with 0 before splitting
X.fillna(0, inplace=True)

In [11]:
X.columns = X.columns.astype(str)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [13]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import MultinomialNB

models = [
    RandomForestClassifier(random_state=42),
    GradientBoostingClassifier(random_state=42),
    AdaBoostClassifier(random_state=42),
    ExtraTreesClassifier(random_state=42),
    BaggingClassifier(random_state=42),
    SVC(random_state=42),
    XGBClassifier(random_state=42),
    CatBoostClassifier(random_state=42, verbose=False),
    LGBMClassifier(random_state=42, verbose=-1),
    MultinomialNB()
]


In [14]:

for model in models:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(model.__class__.__name__)
    print(classification_report(y_test, y_pred))
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("-----------------------------")

RandomForestClassifier
              precision    recall  f1-score   support

       False       0.99      0.96      0.98       353
        True       0.91      0.97      0.94       132

    accuracy                           0.96       485
   macro avg       0.95      0.97      0.96       485
weighted avg       0.97      0.96      0.97       485

Accuracy: 0.9649484536082474
-----------------------------
GradientBoostingClassifier
              precision    recall  f1-score   support

       False       0.96      0.99      0.98       353
        True       0.98      0.90      0.94       132

    accuracy                           0.97       485
   macro avg       0.97      0.95      0.96       485
weighted avg       0.97      0.97      0.97       485

Accuracy: 0.9690721649484536
-----------------------------
AdaBoostClassifier
              precision    recall  f1-score   support

       False       0.96      1.00      0.98       353
        True       0.99      0.90      0.94       

In [15]:
model = SVC(random_state=42)

In [16]:
# model.fit(X_train_vect, y_train)
model.fit(X_train, y_train)

SVC(random_state=42)

In [17]:
y_pred = model.predict(X_test)

In [18]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       1.00      0.98      0.99       353
        True       0.94      0.99      0.97       132

    accuracy                           0.98       485
   macro avg       0.97      0.98      0.98       485
weighted avg       0.98      0.98      0.98       485



In [19]:
import joblib

joblib.dump(model, 'model.pkl')
joblib.dump(X.columns, 'model_features.pkl')

['model_features.pkl']

model = joblib.load('model.pkl')
model_features = joblib.load('model_features.pkl')

# Function to make predictions
def predict_hide_comment(comments, sentiment_category) -> bool:
    # Create a DataFrame for the input data
    input_data = pd.DataFrame({
        # 'question code': [question_code],
        'comments': [comments],
        'sentiment category': [sentiment_category],
    })
    
    # Convert categorical data to numerical data for 'question code'
    input_data = pd.get_dummies(input_data, columns=['sentiment category'])
    
    # Vectorize 'comments' column using the loaded TF-IDF vectorizer

    comments_tfidf = vectorizer.transform(input_data['comments']).toarray()
    
    # Drop the original 'comments' column and add the TF-IDF features
    input_data = input_data.drop(columns=['comments'])
    input_data = pd.concat([input_data, pd.DataFrame(comments_tfidf, index=input_data.index)], axis=1)

    # Reindex input_data to match the columns used during training
    # Fill any missing columns with 0
    input_data = input_data.reindex(columns=model_features, fill_value=0)
    
    input_data.columns = input_data.columns.astype(str)
    
    # Make prediction
    prediction = model.predict(input_data)
    
    return prediction[0]


# Example usage
# question_code = 'dq|change_observed'
comments = "There has been an introduction to AI solutions and tools that have helped with workflow."
sentiment_category = "neutral"

result = predict_hide_comment(comments, sentiment_category)
print(f"Prediction: {result}")